# 植物分类（Plant Seedlings）—— Kaggle 竞赛方案

任务：5 类植物幼苗分类，训练集 500 张（每类 100 张），测试集 378 张，评价指标 Micro F1。

## 运行说明

本 Notebook 分为三个阶段，建议按阶段运行：

| 阶段 | Cell | 内容 | 预计耗时 |
|:---:|:---:|:---|:---:|
| 一 | 1 - 5 | 环境检查、路径发现、数据探索、增强管线 | 约 2 分钟 |
| 二 | 6 - 8 | 模型工厂、训练循环、5 折交叉验证 | 冒烟 3 分钟 / 完整 2 至 5 小时 |
| 三 | 9 - 10 | 融合推理、生成提交文件、结果自检 | 约 1 分钟 |

### 使用步骤
1. 在右侧 Add Input 中挂载植物数据集，目录中应包含 train 和 test。
2. 在 Settings 中开启 Internet（用于下载预训练权重），并选择 GPU。
3. 先运行阶段一，检查数据路径和样本统计是否正常。
4. 将 SMOKE_TEST 设为 True 运行一遍阶段二，确认训练链路无误。
5. 冒烟测试通过后改回 False，正式运行 5 折交叉验证。
6. 运行阶段三，生成提交文件。

### 主要设计
- 使用 ImageNet-22k 预训练模型微调，不从零训练。
- 使用 5 折交叉验证，减小单次划分带来的方差。
- 每轮同时评估原始权重与 EMA 权重，取较优者。
- TTA 使用 D4 对称增强（旋转与翻转），因为叶片没有固定朝向。
- 仅在 GPU 计算能力不低于 7.0 时开启 AMP，避免 P100 上出现 NaN。


---
# 阶段一：环境与数据探索
---


## Cell 1 | 环境检查与依赖安装

确认硬件和库版本，并检查能否联网下载预训练权重。


In [ ]:
import subprocess, sys, os, platform

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input", *pkgs],
                   check=False)

# timm 是预训练模型库，Kaggle 镜像通常自带，这里只做兜底安装
try:
    import timm
    print(f"timm 已存在: {timm.__version__}")
except ImportError:
    print("timm 未安装，正在安装 ...")
    _pip("timm")
    import timm
    print(f"timm 安装完成: {timm.__version__}")

import torch, torchvision, numpy, pandas, sklearn, PIL

print("=" * 64)
print(f"python      : {platform.python_version()}")
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"timm        : {timm.__version__}")
print(f"numpy       : {numpy.__version__}")
print(f"pandas      : {pandas.__version__}")
print(f"sklearn     : {sklearn.__version__}")
print(f"PIL         : {PIL.__version__}")
print("=" * 64)

# ---------- GPU 探测 ----------
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU[{i}] : {p.name} | {p.total_memory / 1024**3:.1f} GB | compute capability {p.major}.{p.minor}")
else:
    print("没有检测到 GPU，请在 Settings 中选择 GPU 后重启会话。")
print(f"CPU 核数 : {os.cpu_count()}")

try:
    import psutil
    print(f"内存     : {psutil.virtual_memory().total / 1024**3:.1f} GB")
except Exception:
    pass

# ---------- 网络探测：决定能否下载预训练权重 ----------
print("\n" + "=" * 64)
print("网络探测：尝试下载一个极小的预训练权重 ...")
NET_OK = False
try:
    _m = timm.create_model("resnet18", pretrained=True)
    del _m
    NET_OK = True
    print("网络正常，可以下载预训练权重")
except Exception as e:
    print("下载失败。")
    print(f"   错误类型: {type(e).__name__}")
    print(f"   错误信息: {str(e)[:400]}")
    print("   -> 请到右侧 Settings -> Internet 打开开关 (On)，然后重新运行本 Cell")
    print("   -> 若该开关是置灰不可点的，说明竞赛禁止联网。")
    print("      请检查网络设置，或改用离线预训练权重。")
print("=" * 64)


## Cell 2 | 全局配置与随机种子

所有超参数集中在这里。默认配置对应 run5（384px、弱增强、三个骨干模型）；其余批次只需修改 img_size 和 models。SMOKE_TEST 用于快速验证训练链路是否正常。


In [ ]:
import os, random, time, json, math, warnings, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

warnings.filterwarnings("ignore")

CONFIG = dict(
    seed=42,
    # ---- 数据 ----
    img_size=384,
    # ---- 训练 ----
    n_folds=5,
    epochs=30,
    batch_size=16,
    num_workers=4,
    lr=3e-4,
    weight_decay=0.05,
    layer_decay=0.75,
    warmup_epochs=3,
    label_smoothing=0.1,
    ema_decay=0.995,
    drop_rate=0.2,
    drop_path_rate=0.1,
    early_stop_patience=10,
    # ---- 模型（run5：384px，弱增强）----
    models=[
        "convnext_small.fb_in22k_ft_in1k",
        "swin_tiny_patch4_window7_224.ms_in22k_ft_in1k",
        "tf_efficientnet_b4.ns_jft_in1k",
    ],
    # ---- 开关 ----
    SMOKE_TEST=False,
    USE_TTA=True,
)

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# P100 为 Pascal 架构，fp16 容易出现 NaN，仅在计算能力不低于 7.0 时开启 AMP
if DEVICE.type == "cuda":
    USE_AMP = torch.cuda.get_device_properties(0).major >= 7
else:
    USE_AMP = False
torch.backends.cudnn.benchmark = True

if CONFIG["SMOKE_TEST"]:
    CONFIG.update(epochs=3, batch_size=16, models=CONFIG["models"][:1], USE_TTA=False)
    print("SMOKE_TEST 已开启：单模型 / 第 1 折 / 3 epoch")

print(f"seed={CONFIG['seed']} | device={DEVICE} | AMP={USE_AMP} | img_size={CONFIG['img_size']} "
      f"| batch={CONFIG['batch_size']} | epochs={CONFIG['epochs']}")


## Cell 3 | 自动发现数据路径

数据目录可能是双层嵌套的，上传到 Kaggle 后嵌套层数也可能变化。因此这里不写死路径，而是根据目录结构特征自动搜索。


In [ ]:
from glob import glob

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

INPUT_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
INPUT_ROOTS = [r for r in INPUT_ROOTS if r.exists()] or [Path(".")]

_CNT_CACHE = {}


def _count_imgs(d):
    d = Path(d)
    key = str(d)
    if key not in _CNT_CACHE:
        _CNT_CACHE[key] = sum(1 for p in d.rglob("*") if p.suffix.lower() in IMG_EXT)
    return _CNT_CACHE[key]


def _is_class_dir(d, min_classes=3, min_per_class=10):
    # 子目录数 >= 3 且每个子目录图片数 >= 10  ->  长得像 train 目录
    if not d.is_dir():
        return False
    subs = [s for s in d.iterdir() if s.is_dir()]
    if len(subs) < min_classes:
        return False
    return all(_count_imgs(s) >= min_per_class for s in subs)


def find_train_dir():
    cands = []
    for root in INPUT_ROOTS:
        for d in root.rglob("*"):
            if d.is_dir() and _is_class_dir(d):
                subs = [s for s in d.iterdir() if s.is_dir()]
                score = (len(subs), sum(_count_imgs(s) for s in subs), -len(d.parts))
                cands.append((score, d))
    if not cands:
        return None
    cands.sort(key=lambda x: x[0], reverse=True)
    return cands[0][1]


def find_test_dir(train_dir, min_imgs=50):
    # 测试目录 = 一堆散装图片、没有子目录、且不在 train 之下
    cands = []
    for root in INPUT_ROOTS:
        for d in root.rglob("*"):
            if not d.is_dir():
                continue
            if train_dir is not None and (d == train_dir or train_dir in d.parents):
                continue
            try:
                if any(s.is_dir() for s in d.iterdir()):
                    continue
            except PermissionError:
                continue
            n = _count_imgs(d)
            if n >= min_imgs:
                cands.append(((n, -len(d.parts)), d))
    if not cands:
        return None
    cands.sort(key=lambda x: x[0], reverse=True)
    return cands[0][1]


TRAIN_DIR = find_train_dir()
TEST_DIR = find_test_dir(TRAIN_DIR)

# 类别名 = train 的子目录名，原样保留空格和连字符（"Common wheat" / "Black-grass"）
CLASS_NAMES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()]) if TRAIN_DIR else []

# 找提交模板：/kaggle/input 下任何带 submission 字样的 csv
TEMPLATE_CSV = None
for root in INPUT_ROOTS:
    hits = sorted((Path(p) for p in glob(str(root / "**" / "*.csv"), recursive=True)
                   if "submission" in Path(p).name.lower()),
                  key=lambda p: len(p.parts))
    if hits:
        TEMPLATE_CSV = hits[0]
        break

print("=" * 64)
print(f"TRAIN_DIR   : {TRAIN_DIR}")
print(f"TEST_DIR    : {TEST_DIR}")
print(f"CLASS_NAMES : {CLASS_NAMES}")
print(f"TEMPLATE_CSV: {TEMPLATE_CSV}")
print("=" * 64)

if TRAIN_DIR is None or TEST_DIR is None:
    print("\n自动查找失败，请检查数据目录：")
    for root in INPUT_ROOTS:
        for d in sorted(root.rglob("*"))[:80]:
            print("   ", d)
    raise SystemExit("路径发现失败，已停止")
print(f"路径发现成功：{len(CLASS_NAMES)} 个类别")


## Cell 4 | 数据探索

数量统计、图片尺寸分布、损坏图检查和样本网格图。


In [ ]:
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor


def collect_records(split_dir, class_names=None):
    # class_names=None 表示测试集：散装目录，无标签
    recs = []
    if class_names is None:
        for p in sorted(Path(split_dir).iterdir()):
            if p.suffix.lower() in IMG_EXT:
                recs.append(dict(path=p, label=-1, class_name=""))
    else:
        for ci, cname in enumerate(class_names):
            for p in sorted((Path(split_dir) / cname).iterdir()):
                if p.suffix.lower() in IMG_EXT:
                    recs.append(dict(path=p, label=ci, class_name=cname))
    return recs


def _probe(rec):
    try:
        with Image.open(rec["path"]) as im:
            w, h = im.size
            # 注意：列名不能叫 "mode"，会和 pandas 的 .mode() 方法冲突
            rec.update(width=w, height=h, img_mode=im.mode, ok=True, err="")
    except Exception as e:
        rec.update(width=0, height=0, img_mode="ERR", ok=False, err=str(e)[:80])
    return rec


with ThreadPoolExecutor(max_workers=8) as ex:
    train_recs = list(ex.map(_probe, collect_records(TRAIN_DIR, CLASS_NAMES)))
    test_recs = list(ex.map(_probe, collect_records(TEST_DIR, None)))

df_train = pd.DataFrame(train_recs)
df_test = pd.DataFrame(test_recs)

print("=" * 64)
print("【训练集】每类数量")
print(df_train.groupby("class_name").size().to_string())
print(f"训练集合计: {len(df_train)}")

print("\n【测试集】")
print(f"测试集合计: {len(df_test)}")

for name, df in [("训练集", df_train), ("测试集", df_test)]:
    side = pd.concat([df.width, df.height], axis=1).max(axis=1)
    print(f"\n【{name}尺寸】min={side.min()}  median={int(side.median())}  max={side.max()}")
    print(f"  正方形占比: {(df.width == df.height).mean():.1%}")

# ---------- 关键诊断：图片尺寸会不会泄漏类别信息 ----------
# 本项目实测：长边跨度可达 58 倍(49px ~ 2840px)。若尺寸分布与类别强相关，
# 模型可能学到"图小又糊 -> 某类"这种捷径，而非真正识别植物。
print("\n" + "=" * 64)
print("【尺寸 vs 类别】长边像素分布（排查尺寸捷径）")
print("=" * 64)
_side = pd.concat([df_train.width, df_train.height], axis=1).max(axis=1)
_tab = df_train.assign(side=_side).groupby("class_name")["side"].describe()
print(_tab[["count", "min", "25%", "50%", "75%", "max"]].astype(int).to_string())

_bins = [0, 100, 500, 99999]
_labels = ["<100px", "100-500px", ">500px"]
_bin_col = pd.cut(_side, bins=_bins, labels=_labels)
_ct = pd.crosstab(df_train.class_name, _bin_col)
print("\n分箱计数：")
print(_ct.to_string())
try:
    from scipy.stats import chi2_contingency
    _chi2, _p, _dof, _ = chi2_contingency(_ct.values)
    print(f"\n卡方检验 p = {_p:.2e}")
    if _p < 0.01:
        print("尺寸与类别显著相关（p < 0.01），可能存在捷径特征。")
        print("    应对：训练时放宽 RandomResizedCrop 的 scale 范围，强制模型对尺度不变。")
except Exception as e:
    print(f"（跳过卡方检验: {e}）")
print(f"长边跨度: max/min = {_side.max() / max(_side.min(), 1):.0f}x")

bad_tr = df_train[~df_train.ok]
bad_te = df_test[~df_test.ok]
print(f"\n【损坏/无法读取】训练集 {len(bad_tr)} 张，测试集 {len(bad_te)} 张")
if len(bad_tr):
    print(bad_tr[["path", "err"]].to_string())
if len(bad_te):
    print(bad_te[["path", "err"]].to_string())

print(f"\n【色彩模式】训练集 {df_train.img_mode.value_counts().to_dict()}")

# ---------- 样本网格图 ----------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def show_grid(df, class_names, n_per_class, fname, title):
    n_cls = len(class_names)
    fig, axes = plt.subplots(n_cls, n_per_class, figsize=(2.2 * n_per_class, 2.5 * n_cls))
    axes = np.atleast_2d(axes)
    rng = np.random.RandomState(CONFIG["seed"])
    for r, cname in enumerate(class_names):
        sub = df[(df.class_name == cname) & df.ok]
        idxs = rng.choice(len(sub), size=min(n_per_class, len(sub)), replace=False)
        for c in range(n_per_class):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(idxs):
                row = sub.iloc[idxs[c]]
                ax.imshow(Image.open(row.path).convert("RGB"))
                ax.set_title(f"{cname}\n{row.width}x{row.height}" if c == 0 else f"{row.width}x{row.height}",
                             fontsize=9)
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(WORK_DIR / fname, dpi=110, bbox_inches="tight")
    plt.close(fig)
    print(f"已保存: {WORK_DIR / fname}")


show_grid(df_train, CLASS_NAMES, 5, "eda_train_samples.png", "训练集样本（每类 5 张）")
show_grid(df_test.assign(class_name="(test)"), ["(test)"], 10, "eda_test_samples.png", "测试集随机样本")


## Cell 5 | 数据管线：Dataset、增强与 DataLoader

针对 500 张图采用弱增强，并可视化增强效果。


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

# ImageNet 统计量（预训练模型的标配）
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def build_train_tf(img_size):
    # 弱增强：本数据集的判别信号有较多来自拍摄环境，强增强会破坏该信号
    return T.Compose([
        T.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
        T.RandomHorizontalFlip(0.5),
        T.RandomVerticalFlip(0.5),                        # 叶片无固定朝向
        T.ColorJitter(0.1, 0.1, 0.1, 0.02),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])


def build_eval_tf(img_size):
    return T.Compose([
        T.Resize(int(img_size * 1.14)),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])


class PlantDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths = list(paths)
        self.labels = list(labels)
        self.tf = tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.labels[i]


def make_loader(paths, labels, tf, shuffle, batch_size=None):
    return DataLoader(
        PlantDataset(paths, labels, tf),
        batch_size=batch_size or CONFIG["batch_size"],
        shuffle=shuffle,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
        drop_last=shuffle,
        persistent_workers=CONFIG["num_workers"] > 0,
    )


# 只保留能正常读取的图（测试集保持原索引，后面要靠它对齐提交 ID）
df_train = df_train[df_train.ok].reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

assert df_train.class_name.nunique() == len(CLASS_NAMES), "类别数与目录数不一致！"

print(f"训练集可用: {len(df_train)} 张 | 类别: {CLASS_NAMES}")
print(f"测试集    : {len(df_test)} 张")

# ---------- 增强效果预览 ----------
tf_preview = build_train_tf(CONFIG["img_size"])
raw = Image.open(df_train.path.iloc[0]).convert("RGB")

fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for k, ax in enumerate(axes.ravel()):
    if k == 0:
        ax.imshow(raw)
        ax.set_title("原图", fontsize=11)
    else:
        t = tf_preview(raw).permute(1, 2, 0).numpy()
        t = t * np.array(STD) + np.array(MEAN)
        ax.imshow(np.clip(t, 0, 1))
        ax.set_title(f"增强 #{k}", fontsize=11)
    ax.axis("off")
fig.suptitle("同一张图的 8 种随机增强结果（确认增强强度是否合理）", fontsize=13)
fig.tight_layout()
fig.savefig(WORK_DIR / "eda_augment_preview.png", dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"已保存: {WORK_DIR / 'eda_augment_preview.png'}")

# ---------- DataLoader 自检 ----------
_ld = make_loader(df_train.path[:32], df_train.label[:32],
                  build_eval_tf(CONFIG["img_size"]), shuffle=False, batch_size=8)
_x, _y = next(iter(_ld))
print(f"\nDataLoader 自检通过  batch={tuple(_x.shape)}  labels={_y.tolist()[:8]}")
print(f"   batch 数值范围 [{_x.min():.2f}, {_x.max():.2f}]（归一化后应大致在 -3 ~ 3）")
del _ld, _x, _y

print("\n" + "=" * 64)
print("阶段一完成，可以继续运行阶段二。")
print("=" * 64)


---
# 阶段二：模型与训练
---

建议先做冒烟测试：将 Cell 2 中的 SMOKE_TEST 设为 True，依次运行 Cell 6、7、8，约 3 分钟。确认没有报错且验证集 F1 上升后，再改回 False 正式训练。


## Cell 6 | 模型工厂

创建预训练骨干，并逐个做一次前向检查。


In [ ]:
import timm


def build_model(name, num_classes=None, pretrained=True):
    num_classes = num_classes or len(CLASS_NAMES)
    kw = dict(pretrained=pretrained, num_classes=num_classes,
              drop_rate=CONFIG["drop_rate"], drop_path_rate=CONFIG["drop_path_rate"])
    try:
        # 显式传 img_size，让 Swin 这类对输入尺寸敏感的结构正确初始化
        return timm.create_model(name, img_size=CONFIG["img_size"], **kw)
    except TypeError:
        return timm.create_model(name, **kw)


print("=" * 74)
for name in CONFIG["models"]:
    try:
        m = build_model(name).to(DEVICE).eval()
        n_param = sum(p.numel() for p in m.parameters()) / 1e6
        with torch.no_grad():
            x = torch.randn(2, 3, CONFIG["img_size"], CONFIG["img_size"], device=DEVICE)
            if USE_AMP:
                with torch.autocast("cuda", dtype=torch.float16):
                    y = m(x)
            else:
                y = m(x)
        print(f"OK   {name:52s} {n_param:6.1f}M  out={tuple(y.shape)}")
        del m, x, y
    except Exception as e:
        print(f"FAIL {name:52s} 创建失败: {type(e).__name__}: {str(e)[:200]}")
print("=" * 74)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\n注：只有完整跑通的模型才会进入 Cell 8 的训练列表。")


## Cell 7 | 训练与验证循环

核心函数为 train_one_fold()。每轮同时评估原始权重和 EMA 权重，取验证集 F1 较高的一组保存。


In [ ]:
from timm.scheduler import CosineLRScheduler
from timm.utils import ModelEmaV2
from sklearn.metrics import f1_score

try:
    from timm.optim import create_optimizer_v2
except ImportError:
    from timm.optim import create_optimizer as create_optimizer_v2


def micro_f1(y_true, y_pred):
    # 多分类单标签下 micro-F1 恒等于 accuracy
    return f1_score(y_true, y_pred, average="micro")


class EarlyStopper:
    def __init__(self, patience):
        self.patience = patience
        self.best = -1.0
        self.bad = 0

    def step(self, score):
        if score > self.best + 1e-6:
            self.best = score
            self.bad = 0
            return False
        self.bad += 1
        return self.bad >= self.patience


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, Y = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        P.append(torch.softmax(logits.float(), -1).cpu())
        Y.append(torch.as_tensor(y))
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


def train_one_fold(model_name, tr_idx, va_idx, fold, epochs, tag=""):
    tr = df_train.iloc[tr_idx].reset_index(drop=True)
    va = df_train.iloc[va_idx].reset_index(drop=True)

    tr_loader = make_loader(tr.path, tr.label, build_train_tf(CONFIG["img_size"]), shuffle=True)
    va_loader = make_loader(va.path, va.label, build_eval_tf(CONFIG["img_size"]),
                            shuffle=False, batch_size=CONFIG["batch_size"] * 2)

    model = build_model(model_name).to(DEVICE)
    ema = ModelEmaV2(model, decay=CONFIG["ema_decay"])

    optimizer = create_optimizer_v2(
        model, opt="adamw", lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"], layer_decay=CONFIG["layer_decay"])
    scheduler = CosineLRScheduler(
        optimizer, t_initial=epochs, lr_min=1e-6,
        warmup_t=CONFIG["warmup_epochs"], warmup_lr_init=1e-6, warmup_prefix=True)

    criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
    stopper = EarlyStopper(CONFIG["early_stop_patience"])
    best = dict(score=-1.0, state=None, which=None, epoch=-1)
    history = []

    for ep in range(epochs):
        scheduler.step(ep)
        model.train()
        tot, n_seen = 0.0, 0

        for x, y in tr_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            ema.update(model)

            tot += loss.item() * x.size(0)
            n_seen += x.size(0)

        p_raw, y_val = evaluate(model, va_loader)
        s_raw = micro_f1(y_val, p_raw.argmax(1))
        p_ema, _ = evaluate(ema.module, va_loader)
        s_ema = micro_f1(y_val, p_ema.argmax(1))
        which, score = ("ema", s_ema) if s_ema >= s_raw else ("raw", s_raw)

        history.append(dict(epoch=ep + 1, loss=tot / max(n_seen, 1),
                            f1_raw=float(s_raw), f1_ema=float(s_ema)))
        improved = score > best["score"] + 1e-6
        print(f"{tag}[F{fold}][{ep + 1:02d}/{epochs}] loss={tot / max(n_seen, 1):.4f} "
              f"val_f1 raw={s_raw:.4f} ema={s_ema:.4f} -> 采纳 {which} {score:.4f}"
              + ("  *" if improved else ""))

        if improved:
            src = ema.module if which == "ema" else model
            best.update(score=score, which=which, epoch=ep + 1,
                        state=copy.deepcopy(src.state_dict()))

        if stopper.step(score):
            print(f"{tag}[F{fold}] 早停于 epoch {ep + 1}（最佳 {stopper.best:.4f} @ epoch {best['epoch']}）")
            break

    del model, ema, optimizer, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best, history


## Cell 8 | 5 折交叉验证主流程

这是最耗时的一步。每折结束后将结果写入 /kaggle/working/，避免会话超时丢失结果。


In [ ]:
from sklearn.model_selection import StratifiedKFold


def tta_views(x, n=8):
    # D4 对称增强：4 个旋转 x (原图 + 水平翻转) = 8 路
    if n <= 1:
        return [x]
    views = []
    for r in range(4):
        xr = torch.rot90(x, r, dims=(2, 3))
        views.append(xr)
        views.append(torch.flip(xr, dims=[3]))
    return views[:n]


@torch.no_grad()
def predict_probs(model, loader, n_tta=8):
    model.eval()
    outs = []
    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        views = tta_views(x, n_tta)
        acc = torch.zeros(x.size(0), len(CLASS_NAMES), device=DEVICE)
        for v in views:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(v)
            acc += torch.softmax(logits.float(), dim=-1)
        outs.append((acc / len(views)).cpu().numpy())
    return np.concatenate(outs, axis=0)


def run_cv(model_name, tag=None):
    tag = tag or model_name.split(".")[0]
    n_split = max(2, CONFIG["n_folds"])
    skf = StratifiedKFold(n_splits=n_split, shuffle=True, random_state=CONFIG["seed"])
    splits = list(skf.split(df_train.path, df_train.label))

    n_tta = 8 if (CONFIG["USE_TTA"] and not CONFIG["SMOKE_TEST"]) else 1
    cv_epochs = 3 if CONFIG["SMOKE_TEST"] else CONFIG["epochs"]

    oof = np.zeros((len(df_train), len(CLASS_NAMES)), dtype=np.float32)
    test_paths = list(df_test.path)
    test_loader = make_loader(test_paths, [-1] * len(test_paths),
                              build_eval_tf(CONFIG["img_size"]), shuffle=False,
                              batch_size=CONFIG["batch_size"] * 2)

    fold_scores, fold_tests = [], []
    for fold, (tr_idx, va_idx) in enumerate(splits):
        if CONFIG["SMOKE_TEST"] and fold >= 1:
            break
        print("\n" + "=" * 74)
        print(f"  模型 {model_name} | Fold {fold + 1}/{n_split} | "
              f"train={len(tr_idx)} val={len(va_idx)} | TTA={n_tta}")
        print("=" * 74)

        t0 = time.time()
        best, _ = train_one_fold(model_name, tr_idx, va_idx, fold, cv_epochs, tag=f"[{tag}]")

        torch.save({"state_dict": best["state"], "model_name": model_name,
                    "class_names": CLASS_NAMES, "val_f1": best["score"],
                    "which": best["which"], "epoch": best["epoch"],
                    "img_size": CONFIG["img_size"]},
                   WORK_DIR / f"{tag}_fold{fold}.pt")

        # 用最佳权重重新推理（验证集填 OOF，测试集做集成）
        model = build_model(model_name, pretrained=False).to(DEVICE)
        model.load_state_dict(best["state"])

        va = df_train.iloc[va_idx].reset_index(drop=True)
        va_loader = make_loader(va.path, va.label, build_eval_tf(CONFIG["img_size"]),
                                shuffle=False, batch_size=CONFIG["batch_size"] * 2)
        p_va = predict_probs(model, va_loader, n_tta)
        p_te = predict_probs(model, test_loader, n_tta)

        oof[va_idx] = p_va
        fold_tests.append(p_te)
        s = micro_f1(va.label, p_va.argmax(1))
        fold_scores.append(float(s))

        print(f">>> Fold {fold + 1} 完成 | val micro-F1 = {s:.4f} | "
              f"最佳 epoch={best['epoch']}({best['which']}) | 耗时 {(time.time() - t0) / 60:.1f} min")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # 每折结束就落盘，Kaggle 会话超时也不丢结果
        np.save(WORK_DIR / f"{tag}_oof.npy", oof)
        np.save(WORK_DIR / f"{tag}_test.npy", np.mean(fold_tests, axis=0))

    if CONFIG["SMOKE_TEST"]:
        print("\nSMOKE_TEST 模式只跑了 1 折，OOF 分数不完整。")
        oof_f1 = float("nan")
    else:
        oof_f1 = float(micro_f1(df_train.label.values, oof.argmax(1)))

    print("\n" + "=" * 74)
    print(f"  {model_name} 交叉验证完成")
    print(f"  各折 micro-F1 : {[round(s, 4) for s in fold_scores]}")
    print(f"  OOF micro-F1  : {oof_f1:.4f}"
          + (f"   (均值 {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f})"
             if fold_scores else ""))
    print("=" * 74)

    json.dump({"model": model_name, "tag": tag, "fold_scores": fold_scores,
               "oof_f1": None if np.isnan(oof_f1) else oof_f1},
              open(WORK_DIR / f"{tag}_cv.json", "w"), indent=2)
    return dict(tag=tag, oof=oof, test=np.mean(fold_tests, axis=0), oof_f1=oof_f1)


# ---------------- 主循环 ----------------
RESULTS = {}
for _name in CONFIG["models"]:
    RESULTS[_name] = run_cv(_name)

print("\n" + "=" * 74)
print("  阶段二完成。各模型 OOF 汇总：")
for _n, _r in RESULTS.items():
    print(f"    {_n:52s} {_r['oof_f1']:.4f}")
print("=" * 74)


---
# 阶段三：融合推理与提交
---


## Cell 9 | 贪心融合与生成提交文件

默认读取 Cell 8 保存的概率（TTA 已在训练时完成）。若要改变 TTA 路数，可将 RECOMPUTE_TTA 设为 True 重新计算，无需重新训练。


In [ ]:
# TTA 推理已在 Cell 8 每折结束时完成并落盘。
# 若要换 TTA 路数重算（秒级，不用重新训练），把下面改成 True
RECOMPUTE_TTA = False
N_TTA = 8      # 1 = 关闭 TTA；8 = 完整 D4 对称


def recompute_from_ckpt(tag, model_name, n_tta):
    skf = StratifiedKFold(n_splits=max(2, CONFIG["n_folds"]), shuffle=True,
                          random_state=CONFIG["seed"])
    splits = list(skf.split(df_train.path, df_train.label))
    oof = np.zeros((len(df_train), len(CLASS_NAMES)), dtype=np.float32)

    test_paths = list(df_test.path)
    test_loader = make_loader(test_paths, [-1] * len(test_paths),
                              build_eval_tf(CONFIG["img_size"]), shuffle=False,
                              batch_size=CONFIG["batch_size"] * 2)

    acc, n_used = None, 0
    for fold, (tr_idx, va_idx) in enumerate(splits):
        ckpt = WORK_DIR / f"{tag}_fold{fold}.pt"
        if not ckpt.exists():
            print(f"  跳过 fold{fold}（无 checkpoint）")
            continue
        model = build_model(model_name, pretrained=False).to(DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["state_dict"])

        va = df_train.iloc[va_idx].reset_index(drop=True)
        va_loader = make_loader(va.path, va.label, build_eval_tf(CONFIG["img_size"]),
                                shuffle=False, batch_size=CONFIG["batch_size"] * 2)
        oof[va_idx] = predict_probs(model, va_loader, n_tta)
        p = predict_probs(model, test_loader, n_tta)
        acc = p if acc is None else acc + p
        n_used += 1
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return oof, acc / max(n_used, 1)


print("=" * 74)
print("  收集各模型结果")
print("=" * 74)
per_model = {}
for name in CONFIG["models"]:
    tag = name.split(".")[0]
    f_oof = WORK_DIR / f"{tag}_oof.npy"
    f_test = WORK_DIR / f"{tag}_test.npy"
    if RECOMPUTE_TTA:
        oof, test = recompute_from_ckpt(tag, name, N_TTA)
    elif f_oof.exists() and f_test.exists():
        oof, test = np.load(f_oof), np.load(f_test)
    else:
        print(f"跳过 {name}（没找到 {tag}_oof.npy，说明该模型还没训练）")
        continue
    f1 = float(micro_f1(df_train.label.values, oof.argmax(1)))
    per_model[name] = dict(oof=oof, test=test, f1=f1, tag=tag)
    print(f"  {name:52s} OOF micro-F1 = {f1:.4f}")

assert per_model, "没有任何可用的模型结果，请先运行 Cell 8。"

# ---------- 贪心前向融合 ----------
names = list(per_model)
P = np.stack([per_model[n]["oof"] for n in names])
T = np.stack([per_model[n]["test"] for n in names])
y_train = df_train.label.values


def greedy_select(P, y, steps=25):
    cur = np.zeros_like(P[0])
    sel = []
    for _ in range(steps):
        best_i, best_v = None, -1.0
        for i in range(len(P)):
            cand = (cur * len(sel) + P[i]) / (len(sel) + 1)
            v = micro_f1(y, cand.argmax(1))
            if v > best_v:
                best_i, best_v = i, v
        cur = (cur * len(sel) + P[best_i]) / (len(sel) + 1)
        sel.append(best_i)
    return cur, sel


final_oof, sel = greedy_select(P, y_train, steps=25)
final_test = np.mean([T[i] for i in sel], axis=0)
f1_ens = float(micro_f1(y_train, final_oof.argmax(1)))
best_name = max(names, key=lambda n: per_model[n]["f1"])
best_f1 = per_model[best_name]["f1"]

from collections import Counter
weights = Counter(names[i] for i in sel)
print("\n" + "=" * 74)
print(f"  贪心融合（{len(names)} 个模型，25 步）OOF micro-F1 = {f1_ens:.4f}")
print(f"  最佳单模型 {best_name} = {best_f1:.4f}")
print("  选出的权重：")
for k, v in weights.most_common():
    print(f"    {k:52s} x{v}")
print("=" * 74)

# ---------- 对齐提交 ID ----------
if TEMPLATE_CSV is not None:
    sub_tpl = pd.read_csv(TEMPLATE_CSV)
    assert list(sub_tpl.columns[:2]) == ["ID", "Category"], f"模板列名异常: {list(sub_tpl.columns)}"
    sub_ids = [str(i) for i in sub_tpl["ID"]]
    print(f"使用提交模板: {TEMPLATE_CSV}（{len(sub_ids)} 行）")
else:
    sub_ids = sorted(p.name for p in Path(TEST_DIR).iterdir() if p.suffix.lower() in IMG_EXT)
    print(f"未找到模板，改用测试目录文件名排序（{len(sub_ids)} 行）")

test_pos = {p.name: i for i, p in enumerate(df_test.path)}
missing = [i for i in sub_ids if i not in test_pos]
assert not missing, f"有 {len(missing)} 个提交 ID 不在测试目录中，例如 {missing[:5]}"

rows = final_test[[test_pos[i] for i in sub_ids]]
pred_idx = rows.argmax(1)
categories = [CLASS_NAMES[i] for i in pred_idx]

submission = pd.DataFrame({"ID": sub_ids, "Category": categories})
out_path = WORK_DIR / "submission-for-task2.csv"
submission.to_csv(out_path, index=False)

print(f"\n已写出 {out_path}")
print(submission.head(8).to_string(index=False))


## Cell 10 | 结果分析与提交自检

混淆矩阵用于分析难分类别。提交前需要确认各项自检全部通过。


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = df_train.label.values
y_pred = final_oof.argmax(1)

print("=" * 74)
print(f"  OOF 最终成绩：micro-F1 = {micro_f1(y_true, y_pred):.4f}")
print("=" * 74)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred)
print("混淆矩阵（行 = 真实，列 = 预测）")
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_string())

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=40, ha="right")
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("预测")
ax.set_ylabel("真实")
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=11,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_title(f"OOF 混淆矩阵 | micro-F1 = {micro_f1(y_true, y_pred):.4f}")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(WORK_DIR / "oof_confusion.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print(f"\n已保存: {WORK_DIR / 'oof_confusion.png'}")

# ---------- 错分样例可视化 ----------
wrong = np.where(y_pred != y_true)[0]
if len(wrong):
    n_show = min(12, len(wrong))
    fig, axes = plt.subplots(2, 6, figsize=(18, 7))
    for k, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if k < n_show:
            i = wrong[k]
            ax.imshow(Image.open(df_train.path.iloc[i]).convert("RGB"))
            conf = final_oof[i].max()
            ax.set_title(f"真: {CLASS_NAMES[y_true[i]]}\n预: {CLASS_NAMES[y_pred[i]]}\n({conf:.2f})",
                         fontsize=8)
    fig.suptitle(f"被错分的验证样本（共 {len(wrong)} 张，展示前 {n_show} 张）", fontsize=13)
    fig.tight_layout()
    fig.savefig(WORK_DIR / "oof_misclassified.png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    print(f"已保存: {WORK_DIR / 'oof_misclassified.png'}")
else:
    print("验证集零错分。")

# ---------- 提交文件自检 ----------
print("\n" + "=" * 74)
print("  提交文件自检")
print("=" * 74)
sub = pd.read_csv(WORK_DIR / "submission-for-task2.csv")
n_pred_classes = sub.Category.nunique()
checks = [
    ("列名恰为 ID,Category", list(sub.columns) == ["ID", "Category"]),
    (f"行数 == {len(sub_ids)}", len(sub) == len(sub_ids)),
    ("没有空值", bool(sub.notna().all().all())),
    ("ID 集合与测试集完全一致", set(sub.ID) == set(sub_ids)),
    ("ID 无重复", bool(sub.ID.is_unique)),
    (f"Category 全部合法", set(sub.Category) <= set(CLASS_NAMES)),
    (f"预测不止一个类别（当前 {n_pred_classes} 类）", n_pred_classes > 1),
]
for name, ok in checks:
    print(f"  {'OK  ' if ok else 'FAIL'} {name}")

all_ok = all(ok for _, ok in checks)
print("\n" + ("全部通过，可以提交。" if all_ok
              else "有检查未通过，请检查输出。"))

print(f"\n预测分布（对比训练集的均衡分布，是否有明显偏斜）：")
print(sub.Category.value_counts().to_string())
print("\n训练集分布（参考）：")
print(df_train.class_name.value_counts().to_string())

print(f"\n提交文件路径: {WORK_DIR / 'submission-for-task2.csv'}")
print("文件位置：/kaggle/working/submission-for-task2.csv")
